In [28]:
import pandas as pd

from sklearn.model_selection import train_test_split

from datasets import Dataset

from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments
)

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support
)

In [29]:
df = pd.read_csv(
    "../datasets/processed/news.csv"
)

In [30]:
df = df[
    ["content", "label"]
]

In [31]:
df = df.sample(
    n=5000,
    random_state=42
)

In [32]:
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df["content"],
    df["label"],
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

In [33]:
tokenizer = DistilBertTokenizerFast.from_pretrained(
    "distilbert-base-uncased"
)

In [34]:
train_encodings = tokenizer(
    train_texts.tolist(),
    truncation=True,
    padding=True,
    max_length=512
)
test_encodings = tokenizer(
    test_texts.tolist(),
    truncation=True,
    padding=True,
    max_length=512
)

In [35]:
train_dataset = Dataset.from_dict({
    "input_ids":
        train_encodings["input_ids"],
    "attention_mask":
        train_encodings["attention_mask"],
    "label":
        train_labels.tolist()
})
test_dataset = Dataset.from_dict({
    "input_ids":
        test_encodings["input_ids"],
    "attention_mask":
        test_encodings["attention_mask"],
    "label":
        test_labels.tolist()
})

In [36]:
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 2145.79it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [37]:
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support
)

def compute_metrics(pred):

    labels = pred.label_ids

    preds = pred.predictions.argmax(-1)

    precision, recall, f1, _ = (
        precision_recall_fscore_support(
            labels,
            preds,
            average="binary"
        )
    )

    acc = accuracy_score(
        labels,
        preds
    )

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [38]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    logging_dir="./logs"
)

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [39]:
from transformers import Trainer

# Ensure your model, datasets, and compute_metrics function are defined before running this
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

c:\Users\asus\OneDrive\Desktop\fake-news-detector\backend\venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.033486,0.007071,0.999000,0.997877,1.000000,0.998937


In [10]:
trainer.evaluate()

NameError: name 'trainer' is not defined

In [8]:
model.save_pretrained(
    "../app/ml/saved_models/distilbert"
)

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.34it/s]


In [9]:
tokenizer.save_pretrained(
    "../app/ml/saved_models/distilbert"
)

('../app/ml/saved_models/distilbert\\tokenizer_config.json',
 '../app/ml/saved_models/distilbert\\tokenizer.json')

In [4]:
import os

os.listdir("./results")

['checkpoint-500']

In [5]:
from transformers import (
    DistilBertForSequenceClassification,
    DistilBertTokenizerFast
)

model = DistilBertForSequenceClassification.from_pretrained(
    "./results/checkpoint-500"
)

tokenizer = DistilBertTokenizerFast.from_pretrained(
    "distilbert-base-uncased"
)

c:\Users\asus\OneDrive\Desktop\fake-news-detector\backend\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 104/104 [00:00<00:00, 2518.10it/s]


In [6]:
model.save_pretrained(
    "../app/ml/saved_models/distilbert"
)

tokenizer.save_pretrained(
    "../app/ml/saved_models/distilbert"
)

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.52it/s]


('../app/ml/saved_models/distilbert\\tokenizer_config.json',
 '../app/ml/saved_models/distilbert\\tokenizer.json')